<a href="https://colab.research.google.com/github/dkngit55-glitch/DKNGIT5/blob/main/mail_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import imaplib
import email
from email.header import decode_header

# 1. 최신 라이브러리 설치 및 임포트
try:
    from google import genai
except ImportError:
    !pip install -q -U google-genai
    from google import genai

from google.colab import userdata

# ==========================================
# 2. 설정 정보
# ==========================================
NAVER_ID = "dkngit5"
NAVER_PW = "HCSHSG79ZBY8"

# Colab Secrets에서 API 키 가져오기
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=API_KEY)
except Exception:
    print("❌ 에러: Colab Secrets에 'GEMINI_API_KEY'를 설정해주세요.")
    client = None

# ==========================================
# 3. AI 요약 함수 (최신 SDK 버전)
# ==========================================
def summarize_with_ai(text, subject):
    if client is None:
        return "AI 클라이언트가 설정되지 않았습니다."

    if not text or len(text.strip()) < 10:
        return "본문 내용이 너무 짧아 요약할 수 없습니다."

    try:
        prompt = f"""
        이메일 제목: {subject}
        이메일 내용: {text}

        위 이메일의 핵심 내용을 3줄 이내로 간결하게 한국어로 요약해줘.
        """

        # 최신 모델인 gemini-1.5-flash 사용 (혹은 gemini-1.5-flash)
        response = client.models.generate_content(
            model="gemini-1.5-flash",
            contents=prompt
        )
        return response.text.strip()
    except Exception as e:
        return f"AI 요약 실패: {e}"

# ==========================================
# 4. 메일 가져오기 및 실행 로직
# ==========================================
def get_latest_email_and_summarize():
    try:
        mail = imaplib.IMAP4_SSL("imap.naver.com")
        mail.login(f"{NAVER_ID}@naver.com", NAVER_PW)
        mail.select("inbox")

        status, messages = mail.search(None, "ALL")
        email_ids = messages[0].split()

        if not email_ids:
            print("메일함이 비어 있습니다.")
            return

        last_email_id = email_ids[-1]
        status, msg_data = mail.fetch(last_email_id, "(RFC822)")

        for response_part in msg_data:
            if isinstance(response_part, tuple):
                msg = email.message_from_bytes(response_part[1])

                # 제목 디코딩
                subject_header = decode_header(msg["Subject"])[0]
                subject = subject_header[0]
                if isinstance(subject, bytes):
                    subject = subject.decode(subject_header[1] if subject_header[1] else "utf-8", errors="ignore")

                # 보낸 사람 디코딩
                from_header = decode_header(msg.get("From"))[0]
                from_res = from_header[0]
                if isinstance(from_res, bytes):
                    from_res = from_res.decode(from_header[1] if from_header[1] else "utf-8", errors="ignore")

                # 본문 추출
                body = ""
                if msg.is_multipart():
                    for part in msg.walk():
                        if part.get_content_type() == "text/plain":
                            charset = part.get_content_charset() or "utf-8"
                            body = part.get_payload(decode=True).decode(charset, errors="ignore")
                            break
                else:
                    charset = msg.get_content_charset() or "utf-8"
                    body = msg.get_payload(decode=True).decode(charset, errors="ignore")

                print("="*60)
                print(f"📧 보낸 사람: {from_res}")
                print(f"📌 제목: {subject}")
                print("-" * 60)

                print("🤖 AI 요약 중...")
                summary = summarize_with_ai(body, subject)
                print(f"📝 요약 결과:\n{summary}")
                print("="*60)

        mail.logout()
    except Exception as e:
        print(f"❌ 오류 발생: {e}")

get_latest_email_and_summarize()

📧 보낸 사람: 이호준
📌 제목: Test1
------------------------------------------------------------
🤖 AI 요약 중...
📝 요약 결과:
AI 요약 실패: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
